In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import logging
import platform
import time
from datetime import datetime

import pandas as pd


# ============================================================
# 0. PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name != "fruitnet-chili-yield":
    if PROJECT_ROOT.parent.name == "fruitnet-chili-yield":
        PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

assert PROJECT_ROOT.name == "fruitnet-chili-yield", (
    f"Please run this script from fruitnet-chili-yield root. Current: {PROJECT_ROOT}"
)

os.chdir(PROJECT_ROOT)


# ============================================================
# 1. WSL / WINDOWS F DRIVE DATA + OUTPUT PATHS
# ============================================================

def get_default_data_root() -> Path:
    """
    Dataset root.

    Default WSL path:
        /mnt/f/fruitnet-chili-yield-data

    Equivalent Windows path:
        F:\\fruitnet-chili-yield-data

    Override:
        export FRUITNET_DATA_ROOT=/mnt/f/your/custom/path
    """
    env_path = os.environ.get("FRUITNET_DATA_ROOT", "").strip()
    if env_path:
        return Path(env_path).expanduser()

    system_name = platform.system().lower()

    if system_name == "linux" and Path("/mnt/f").exists():
        return Path("/mnt/f/fruitnet-chili-yield-data")

    if system_name == "windows":
        return Path("F:/fruitnet-chili-yield-data")

    return PROJECT_ROOT / "data"


def get_default_output_root() -> Path:
    """
    Training outputs root.

    Default WSL path:
        /mnt/f/fruitnet-chili-yield-outputs

    This keeps checkpoints, logs, metrics, runs, and artifacts on F drive.

    Override:
        export FRUITNET_OUTPUT_ROOT=/mnt/f/your/custom/output/path
    """
    env_path = os.environ.get("FRUITNET_OUTPUT_ROOT", "").strip()
    if env_path:
        return Path(env_path).expanduser()

    system_name = platform.system().lower()

    if system_name == "linux" and Path("/mnt/f").exists():
        return Path("/mnt/f/fruitnet-chili-yield-outputs")

    if system_name == "windows":
        return Path("F:/fruitnet-chili-yield-outputs")

    return PROJECT_ROOT


DATA_ROOT = get_default_data_root().resolve()
OUTPUT_ROOT = get_default_output_root().resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"

COCO_RAW = RAW_ROOT / "coco2017"
COCO_PROC = PROCESSED_ROOT / "coco_yolo"

COCO_TRAIN_IMAGES = COCO_PROC / "images" / "train2017"
COCO_VAL_IMAGES = COCO_PROC / "images" / "val2017"
COCO_TRAIN_LABELS = COCO_PROC / "labels" / "train2017"
COCO_VAL_LABELS = COCO_PROC / "labels" / "val2017"

LOG_DIR = OUTPUT_ROOT / "logs"
CONFIG_DATA_DIR = PROJECT_ROOT / "configs" / "data"

RUNS_DIR = OUTPUT_ROOT / "runs" / "detectors" / "coco_pretrain"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "pretrained"
RESULT_DIR = OUTPUT_ROOT / "results" / "exp00_coco_pretrain"
MARKER_DIR = RESULT_DIR / "job_markers"
METRICS_DIR = RESULT_DIR / "metrics_snapshots"

for d in [
    LOG_DIR,
    CONFIG_DATA_DIR,
    RUNS_DIR,
    ARTIFACT_DIR,
    RESULT_DIR,
    MARKER_DIR,
    METRICS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. SAFE WRITE HELPERS
# ============================================================

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding=encoding)
    os.replace(tmp, path)


def atomic_write_json(path: Path, data: dict) -> None:
    atomic_write_text(path, json.dumps(data, indent=2, ensure_ascii=False) + "\n")


def atomic_write_df(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def append_jsonl_fsync(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(payload, ensure_ascii=False) + "\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()
        os.fsync(f.fileno())


# ============================================================
# 3. LOGGING
# ============================================================

LOG_PATH = LOG_DIR / f"02_coco_pretrain_resume_safe_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"


class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()


_log_fp = open(LOG_PATH, "a", encoding="utf-8")
sys.stdout = Tee(sys.__stdout__, _log_fp)
sys.stderr = Tee(sys.__stderr__, _log_fp)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, encoding="utf-8"),
        logging.StreamHandler(sys.__stdout__),
    ],
)

logging.info("PROJECT_ROOT=%s", PROJECT_ROOT)
logging.info("DATA_ROOT=%s", DATA_ROOT)
logging.info("OUTPUT_ROOT=%s", OUTPUT_ROOT)
logging.info("COCO_PROC=%s", COCO_PROC)
logging.info("RUNS_DIR=%s", RUNS_DIR)
logging.info("ARTIFACT_DIR=%s", ARTIFACT_DIR)
logging.info("RESULT_DIR=%s", RESULT_DIR)
logging.info("Python=%s", sys.version)
logging.info("Platform=%s", platform.platform())
logging.info("Log file=%s", LOG_PATH)


# ============================================================
# 4. COCO YAML FOR ULTRALYTICS
# ============================================================

COCO_YAML = CONFIG_DATA_DIR / "coco.yaml"

COCO_80_NAMES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "airplane",
    5: "bus",
    6: "train",
    7: "truck",
    8: "boat",
    9: "traffic light",
    10: "fire hydrant",
    11: "stop sign",
    12: "parking meter",
    13: "bench",
    14: "bird",
    15: "cat",
    16: "dog",
    17: "horse",
    18: "sheep",
    19: "cow",
    20: "elephant",
    21: "bear",
    22: "zebra",
    23: "giraffe",
    24: "backpack",
    25: "umbrella",
    26: "handbag",
    27: "tie",
    28: "suitcase",
    29: "frisbee",
    30: "skis",
    31: "snowboard",
    32: "sports ball",
    33: "kite",
    34: "baseball bat",
    35: "baseball glove",
    36: "skateboard",
    37: "surfboard",
    38: "tennis racket",
    39: "bottle",
    40: "wine glass",
    41: "cup",
    42: "fork",
    43: "knife",
    44: "spoon",
    45: "bowl",
    46: "banana",
    47: "apple",
    48: "sandwich",
    49: "orange",
    50: "broccoli",
    51: "carrot",
    52: "hot dog",
    53: "pizza",
    54: "donut",
    55: "cake",
    56: "chair",
    57: "couch",
    58: "potted plant",
    59: "bed",
    60: "dining table",
    61: "toilet",
    62: "tv",
    63: "laptop",
    64: "mouse",
    65: "remote",
    66: "keyboard",
    67: "cell phone",
    68: "microwave",
    69: "oven",
    70: "toaster",
    71: "sink",
    72: "refrigerator",
    73: "book",
    74: "clock",
    75: "vase",
    76: "scissors",
    77: "teddy bear",
    78: "hair drier",
    79: "toothbrush",
}


def load_coco_names() -> dict:
    names_json = COCO_PROC / "coco_names.json"

    if names_json.exists():
        try:
            data = json.loads(names_json.read_text(encoding="utf-8"))
            return {int(k): str(v) for k, v in data.items()}
        except Exception as e:
            logging.warning("Failed to read %s. Fallback to COCO_80_NAMES. Error=%s", names_json, repr(e))

    return COCO_80_NAMES


def write_coco_yaml() -> None:
    names = load_coco_names()
    names_yaml = "\n".join([f"  {int(k)}: {v}" for k, v in sorted(names.items())])

    yaml_text = f"""
path: {COCO_PROC.as_posix()}
train: images/train2017
val: images/val2017
test: images/val2017
names:
{names_yaml}
""".strip() + "\n"

    atomic_write_text(COCO_YAML, yaml_text)
    logging.info("Wrote COCO YAML: %s", COCO_YAML)


write_coco_yaml()


# ============================================================
# 5. DATASET CHECK
# ============================================================

def check_dataset_ready() -> None:
    missing = []

    for p in [
        COCO_PROC,
        COCO_TRAIN_IMAGES,
        COCO_VAL_IMAGES,
        COCO_TRAIN_LABELS,
        COCO_VAL_LABELS,
    ]:
        if not p.exists():
            missing.append(str(p))

    if missing:
        msg = (
            "\nCOCO YOLO dataset is missing on F drive.\n\n"
            "Expected dataset root:\n"
            f"  {COCO_PROC}\n\n"
            "Missing paths:\n"
            + "\n".join([f"  - {m}" for m in missing])
            + "\n\nRun dataset preparation first:\n"
            "  python 01_download_prepare_datasets_wsl_f_drive.py\n\n"
            "Or set correct path:\n"
            "  export FRUITNET_DATA_ROOT=/mnt/f/fruitnet-chili-yield-data\n"
        )
        raise FileNotFoundError(msg)

    n_train_img = len(list(COCO_TRAIN_IMAGES.glob("*")))
    n_val_img = len(list(COCO_VAL_IMAGES.glob("*")))
    n_train_lab = len(list(COCO_TRAIN_LABELS.glob("*.txt")))
    n_val_lab = len(list(COCO_VAL_LABELS.glob("*.txt")))

    logging.info("COCO train images=%s labels=%s", n_train_img, n_train_lab)
    logging.info("COCO val images=%s labels=%s", n_val_img, n_val_lab)

    print("COCO dataset ready:")
    print("  DATA_ROOT:", DATA_ROOT)
    print("  COCO_PROC:", COCO_PROC)
    print("  train images:", n_train_img)
    print("  val images:", n_val_img)
    print("  train labels:", n_train_lab)
    print("  val labels:", n_val_lab)

    if n_train_img == 0 or n_val_img == 0:
        raise RuntimeError("COCO image folders exist but contain no images.")


check_dataset_ready()


# ============================================================
# 6. TRAINING CONFIG
# ============================================================

IMG_SIZE = int(os.environ.get("IMG_SIZE", 640))
BATCH = int(os.environ.get("BATCH", 8))
EPOCHS = int(os.environ.get("EPOCHS", 50))
WORKERS = int(os.environ.get("WORKERS", 8))
SEED = int(os.environ.get("SEED", 42))
PATIENCE = int(os.environ.get("PATIENCE", 20))
SAVE_PERIOD = int(os.environ.get("SAVE_PERIOD", 1))

RUN_TRAINING = os.environ.get("RUN_TRAINING", "1").strip() not in ["0", "false", "False", "no", "NO"]
FORCE_RETRAIN_DONE = os.environ.get("FORCE_RETRAIN_DONE", "0").strip() in ["1", "true", "True", "yes", "YES"]

DEVICE_ENV = os.environ.get("DEVICE", "").strip()

PRETRAIN_JOBS = [
    {
        "model_key": "yolov5s",
        "model_uri": "yolov5su.pt",
        "note": "YOLOv5 small baseline through Ultralytics unified weights.",
    },
    {
        "model_key": "yolov8n",
        "model_uri": "yolov8n.pt",
        "note": "YOLOv8 nano baseline.",
    },
    {
        "model_key": "fruitnet_b",
        "model_uri": "configs/models/fruitnet_b.yaml",
        "note": "Internal baseline: compact YOLOv8-style detector without GhostConv, Coordinate Attention, or SIoU.",
    },
    {
        "model_key": "fruitnet_g",
        "model_uri": "configs/models/fruitnet_g.yaml",
        "note": "FruitNet-B + GhostConv.",
    },
    {
        "model_key": "fruitnet_gc",
        "model_uri": "configs/models/fruitnet_gc.yaml",
        "note": "FruitNet-G + Coordinate Attention.",
    },
    {
        "model_key": "fruitnet_gcs",
        "model_uri": "configs/models/fruitnet_gcs.yaml",
        "note": "FruitNet-GC + SIoU math prepared.",
    },
]


# ============================================================
# 7. IMPORT TORCH / ULTRALYTICS / FRUITNET MODULES
# ============================================================

import torch
from ultralytics import YOLO

try:
    from models.fruitnet.register_ultralytics_modules import register_fruitnet_modules

    register_fruitnet_modules()
    logging.info("FruitNet custom Ultralytics modules registered.")
    print("FruitNet custom modules registered.")
except Exception as e:
    logging.warning("Could not register FruitNet custom modules: %s", repr(e))
    print("Warning: Could not register FruitNet custom modules.")
    print("If you train fruitnet_*.yaml models, make sure models/fruitnet/register_ultralytics_modules.py exists.")
    print("Error:", repr(e))

logging.info("torch=%s cuda=%s", torch.__version__, torch.cuda.is_available())

if DEVICE_ENV:
    DEVICE = DEVICE_ENV
else:
    DEVICE = 0 if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    logging.info("GPU=%s", torch.cuda.get_device_name(0))
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Training will use CPU unless DEVICE is changed.")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("COCO_YAML:", COCO_YAML.resolve())
print("RUNS_DIR:", RUNS_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("LOG_PATH:", LOG_PATH)
print("RUN_TRAINING:", RUN_TRAINING)
print("FORCE_RETRAIN_DONE:", FORCE_RETRAIN_DONE)
print("EPOCHS:", EPOCHS)
print("BATCH:", BATCH)
print("IMG_SIZE:", IMG_SIZE)
print("WORKERS:", WORKERS)
print("PATIENCE:", PATIENCE)
print("SAVE_PERIOD:", SAVE_PERIOD)
print("DEVICE:", DEVICE)


# ============================================================
# 8. HELPER FUNCTIONS
# ============================================================

PROGRESS_PATH = RESULT_DIR / "coco_pretrain_progress.csv"
SUMMARY_PATH = RESULT_DIR / "coco_pretrain_summary.csv"
STATUS_PATH = RESULT_DIR / "coco_pretrain_status.json"


def find_best_and_last(run_dir: Path):
    weights_dir = run_dir / "weights"
    best = weights_dir / "best.pt"
    last = weights_dir / "last.pt"
    return best if best.exists() else None, last if last.exists() else None


def copy_if_exists(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src and src.exists():
        shutil.copy2(src, dst)
        return str(dst)
    return ""


def model_uri_exists_or_builtin(model_uri: str) -> bool:
    lower = model_uri.lower()
    if lower.endswith(".pt"):
        return True
    return Path(model_uri).exists()


def should_use_pretrained(model_uri: str) -> bool:
    return model_uri.lower().endswith(".pt")


def is_finished_resume_error(e: Exception) -> bool:
    s = repr(e).lower()
    return "finished" in s and "nothing to resume" in s


def scalarize(value):
    try:
        if hasattr(value, "item"):
            return float(value.item())
        if isinstance(value, (int, float, str, bool)):
            return value
        return str(value)
    except Exception:
        return str(value)


def collect_trainer_metrics(trainer) -> dict:
    payload = {}
    for attr in ["epoch", "best_fitness", "fitness"]:
        try:
            payload[attr] = scalarize(getattr(trainer, attr))
        except Exception:
            pass

    for attr in ["metrics", "loss_items", "tloss"]:
        try:
            obj = getattr(trainer, attr)
            if isinstance(obj, dict):
                payload[attr] = {str(k): scalarize(v) for k, v in obj.items()}
            else:
                payload[attr] = scalarize(obj)
        except Exception:
            pass

    return payload


def add_crash_safe_callbacks(model, context: dict) -> None:
    def callback(event_name: str):
        def _cb(trainer):
            payload = {
                "time": now_iso(),
                "event": event_name,
                "model_key": context["model_key"],
                "run_dir": str(context["run_dir"]),
                "last_pt": str(context["last_pt"]),
                "best_pt": str(context["best_pt"]),
            }
            payload.update(collect_trainer_metrics(trainer))
            append_jsonl_fsync(context["events_jsonl"], payload)
            atomic_write_json(context["heartbeat_json"], payload)
        return _cb

    for event_name in ["on_train_epoch_end", "on_fit_epoch_end", "on_model_save", "on_train_end"]:
        try:
            model.add_callback(event_name, callback(event_name))
        except Exception as e:
            logging.warning("Could not add callback %s for %s: %s", event_name, context["model_key"], repr(e))


def snapshot_training_metrics(model_key: str, run_dir: Path) -> dict:
    snapshot = {
        "model_key": model_key,
        "run_dir": str(run_dir),
        "results_csv": "",
        "args_yaml": "",
        "copied_files": [],
    }

    dst_dir = METRICS_DIR / model_key
    dst_dir.mkdir(parents=True, exist_ok=True)

    for fname in ["results.csv", "args.yaml"]:
        src = run_dir / fname
        if src.exists():
            dst = dst_dir / fname
            shutil.copy2(src, dst)
            snapshot[fname.replace(".", "_")] = str(dst)
            snapshot["copied_files"].append(str(dst))

    for p in run_dir.glob("*.png"):
        dst = dst_dir / p.name
        shutil.copy2(p, dst)
        snapshot["copied_files"].append(str(dst))

    return snapshot


def load_existing_progress() -> dict:
    if not PROGRESS_PATH.exists():
        return {}
    try:
        df = pd.read_csv(PROGRESS_PATH)
        if "model_key" not in df.columns:
            return {}
        return {str(r["model_key"]): r.to_dict() for _, r in df.iterrows()}
    except Exception as e:
        logging.warning("Could not read existing progress CSV: %s", repr(e))
        return {}


def build_initial_rows() -> list[dict]:
    existing = load_existing_progress()
    rows = []
    for job in PRETRAIN_JOBS:
        model_key = job["model_key"]
        run_dir = RUNS_DIR / model_key
        best, last = find_best_and_last(run_dir)
        exported = ARTIFACT_DIR / f"{model_key}_coco_best.pt"
        done_marker = MARKER_DIR / f"{model_key}.done.json"
        heartbeat = MARKER_DIR / f"{model_key}.heartbeat.json"
        events_jsonl = MARKER_DIR / f"{model_key}.events.jsonl"

        row = {
            "model_key": model_key,
            "model_uri": job["model_uri"],
            "run_dir": str(run_dir),
            "status": "pending",
            "best_pt": str(best) if best else "",
            "last_pt": str(last) if last else "",
            "exported_best": str(exported) if exported.exists() else "",
            "resume_from": "",
            "done_marker": str(done_marker),
            "heartbeat_json": str(heartbeat),
            "events_jsonl": str(events_jsonl),
            "epochs": EPOCHS,
            "batch": BATCH,
            "imgsz": IMG_SIZE,
            "workers": WORKERS,
            "device": DEVICE,
            "seed": SEED,
            "patience": PATIENCE,
            "save_period": SAVE_PERIOD,
            "data_yaml": str(COCO_YAML.resolve()),
            "data_root": str(DATA_ROOT),
            "coco_proc": str(COCO_PROC),
            "started_at": "",
            "finished_at": "",
            "error": "",
            "note": job.get("note", ""),
        }

        if model_key in existing:
            old = existing[model_key]
            for k, v in old.items():
                if k in row and pd.notna(v):
                    row[k] = v

            # Always refresh live file paths in case OUTPUT_ROOT changed.
            row["run_dir"] = str(run_dir)
            row["best_pt"] = str(best) if best else row.get("best_pt", "")
            row["last_pt"] = str(last) if last else row.get("last_pt", "")
            row["exported_best"] = str(exported) if exported.exists() else row.get("exported_best", "")
            row["done_marker"] = str(done_marker)
            row["heartbeat_json"] = str(heartbeat)
            row["events_jsonl"] = str(events_jsonl)

        rows.append(row)

    return rows


def save_progress(rows: list[dict]) -> None:
    df = pd.DataFrame(rows)
    atomic_write_df(PROGRESS_PATH, df)
    atomic_write_df(SUMMARY_PATH, df)


def update_row(rows: list[dict], model_key: str, **updates) -> dict:
    for row in rows:
        if row["model_key"] == model_key:
            row.update(updates)
            save_progress(rows)
            return row
    raise KeyError(model_key)


def done_marker_is_valid(row: dict) -> bool:
    if FORCE_RETRAIN_DONE:
        return False

    marker = Path(str(row["done_marker"]))
    exported = Path(str(row.get("exported_best", ""))) if str(row.get("exported_best", "")).strip() else None

    return marker.exists() and exported is not None and exported.exists()


# ============================================================
# 9. TRAIN ONE MODEL WITH AUTO RESUME
# ============================================================

def train_one_coco_job(job: dict, rows: list[dict]) -> dict:
    model_key = job["model_key"]
    model_uri = job["model_uri"]
    run_dir = RUNS_DIR / model_key
    done_marker = MARKER_DIR / f"{model_key}.done.json"
    heartbeat_json = MARKER_DIR / f"{model_key}.heartbeat.json"
    events_jsonl = MARKER_DIR / f"{model_key}.events.jsonl"

    row = next(r for r in rows if r["model_key"] == model_key)

    best, last = find_best_and_last(run_dir)
    exported = ARTIFACT_DIR / f"{model_key}_coco_best.pt"

    row.update({
        "run_dir": str(run_dir),
        "best_pt": str(best) if best else "",
        "last_pt": str(last) if last else "",
        "exported_best": str(exported) if exported.exists() else row.get("exported_best", ""),
        "done_marker": str(done_marker),
        "heartbeat_json": str(heartbeat_json),
        "events_jsonl": str(events_jsonl),
        "error": "",
    })
    save_progress(rows)

    if done_marker_is_valid(row):
        logging.info("Skipping completed job %s because done marker exists: %s", model_key, done_marker)
        update_row(rows, model_key, status="skipped_completed", finished_at=now_iso())
        return row

    if not RUN_TRAINING:
        logging.info("RUN_TRAINING=False. Skipping training for %s.", model_key)
        update_row(rows, model_key, status="skipped_run_training_false", finished_at=now_iso())
        return row

    if not model_uri_exists_or_builtin(model_uri):
        err = f"Model config not found: {model_uri}"
        logging.error(err)
        update_row(rows, model_key, status="failed", error=err, finished_at=now_iso())
        return row

    started_at = now_iso()
    resume_from = str(last) if last and last.exists() else ""

    update_row(
        rows,
        model_key,
        status="running_resume" if resume_from else "running_new",
        started_at=started_at,
        resume_from=resume_from,
        error="",
    )

    start_payload = {
        "time": started_at,
        "event": "job_start",
        "model_key": model_key,
        "model_uri": model_uri,
        "run_dir": str(run_dir),
        "resume_from": resume_from,
        "data_yaml": str(COCO_YAML.resolve()),
        "output_root": str(OUTPUT_ROOT),
    }
    append_jsonl_fsync(events_jsonl, start_payload)
    atomic_write_json(heartbeat_json, start_payload)

    context = {
        "model_key": model_key,
        "run_dir": run_dir,
        "best_pt": run_dir / "weights" / "best.pt",
        "last_pt": run_dir / "weights" / "last.pt",
        "heartbeat_json": heartbeat_json,
        "events_jsonl": events_jsonl,
    }

    try:
        if resume_from:
            logging.info("Resuming %s from %s", model_key, resume_from)
            model = YOLO(resume_from)
            add_crash_safe_callbacks(model, context)

            # Exact resume: Ultralytics restores optimizer, scheduler, epoch, and original args.
            model.train(resume=True)
        else:
            logging.info("Starting new training job %s from %s", model_key, model_uri)
            model = YOLO(model_uri)
            add_crash_safe_callbacks(model, context)

            model.train(
                data=str(COCO_YAML),
                epochs=EPOCHS,
                imgsz=IMG_SIZE,
                batch=BATCH,
                workers=WORKERS,
                device=DEVICE,
                seed=SEED,
                patience=PATIENCE,
                project=str(RUNS_DIR),
                name=model_key,
                exist_ok=True,
                verbose=True,
                pretrained=should_use_pretrained(model_uri),
                plots=True,
                save=True,
                save_period=SAVE_PERIOD,
            )

        best, last = find_best_and_last(run_dir)
        exported_path = ""
        if best:
            exported_path = copy_if_exists(best, exported)

        metrics_snapshot = snapshot_training_metrics(model_key, run_dir)

        done_payload = {
            "time": now_iso(),
            "event": "job_done",
            "model_key": model_key,
            "model_uri": model_uri,
            "run_dir": str(run_dir),
            "best_pt": str(best) if best else "",
            "last_pt": str(last) if last else "",
            "exported_best": exported_path,
            "metrics_snapshot": metrics_snapshot,
        }
        atomic_write_json(done_marker, done_payload)
        append_jsonl_fsync(events_jsonl, done_payload)
        atomic_write_json(heartbeat_json, done_payload)

        update_row(
            rows,
            model_key,
            status="ok",
            best_pt=str(best) if best else "",
            last_pt=str(last) if last else "",
            exported_best=exported_path,
            finished_at=now_iso(),
            error="",
        )

        logging.info("Finished training %s", model_key)

    except Exception as e:
        best, last = find_best_and_last(run_dir)

        if is_finished_resume_error(e) and (best or last):
            # This happens when an old run completed but no done marker was present.
            exported_path = ""
            if best:
                exported_path = copy_if_exists(best, exported)
            metrics_snapshot = snapshot_training_metrics(model_key, run_dir)

            done_payload = {
                "time": now_iso(),
                "event": "job_already_finished_detected",
                "model_key": model_key,
                "run_dir": str(run_dir),
                "best_pt": str(best) if best else "",
                "last_pt": str(last) if last else "",
                "exported_best": exported_path,
                "metrics_snapshot": metrics_snapshot,
                "resume_error": repr(e),
            }
            atomic_write_json(done_marker, done_payload)
            append_jsonl_fsync(events_jsonl, done_payload)
            atomic_write_json(heartbeat_json, done_payload)

            update_row(
                rows,
                model_key,
                status="ok_already_finished",
                best_pt=str(best) if best else "",
                last_pt=str(last) if last else "",
                exported_best=exported_path,
                finished_at=now_iso(),
                error="",
            )
            logging.info("Marked %s as already finished after resume message.", model_key)
        else:
            err = repr(e)
            fail_payload = {
                "time": now_iso(),
                "event": "job_failed",
                "model_key": model_key,
                "run_dir": str(run_dir),
                "best_pt": str(best) if best else "",
                "last_pt": str(last) if last else "",
                "error": err,
            }
            append_jsonl_fsync(events_jsonl, fail_payload)
            atomic_write_json(heartbeat_json, fail_payload)

            update_row(
                rows,
                model_key,
                status="failed",
                best_pt=str(best) if best else "",
                last_pt=str(last) if last else "",
                finished_at=now_iso(),
                error=err,
            )
            logging.exception("COCO pretrain failed for %s", model_key)

    return next(r for r in rows if r["model_key"] == model_key)


# ============================================================
# 10. RUN ALL PRETRAIN JOBS
# ============================================================

rows = build_initial_rows()
save_progress(rows)

master_status = {
    "time": now_iso(),
    "event": "script_start",
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "coco_proc": str(COCO_PROC),
    "coco_yaml": str(COCO_YAML.resolve()),
    "runs_dir": str(RUNS_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "result_dir": str(RESULT_DIR),
    "progress_csv": str(PROGRESS_PATH),
    "summary_csv": str(SUMMARY_PATH),
    "log_file": str(LOG_PATH),
    "device": str(DEVICE),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "run_training": RUN_TRAINING,
    "force_retrain_done": FORCE_RETRAIN_DONE,
    "epochs": EPOCHS,
    "batch": BATCH,
    "imgsz": IMG_SIZE,
    "workers": WORKERS,
    "patience": PATIENCE,
    "save_period": SAVE_PERIOD,
}
atomic_write_json(STATUS_PATH, master_status)

for job in PRETRAIN_JOBS:
    row = train_one_coco_job(job, rows)
    print("\nLatest result:")
    print(pd.DataFrame([row]))
    time.sleep(1)


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

save_progress(rows)

df = pd.DataFrame(rows)

print("\nSummary:", SUMMARY_PATH)
print(
    df[
        [
            "model_key",
            "status",
            "resume_from",
            "best_pt",
            "last_pt",
            "exported_best",
            "error",
        ]
    ]
)

for _, r in df.iterrows():
    exported = str(r.get("exported_best", "")).strip()
    if exported:
        print(r["model_key"], "exported file exists:", Path(exported).exists(), exported)

master_status.update(
    {
        "time": now_iso(),
        "event": "script_end",
        "summary_csv": str(SUMMARY_PATH),
        "progress_csv": str(PROGRESS_PATH),
        "log_file": str(LOG_PATH),
    }
)
atomic_write_json(STATUS_PATH, master_status)

print("\nSaved:")
print(" -", PROGRESS_PATH)
print(" -", SUMMARY_PATH)
print(" -", STATUS_PATH)
print(" -", LOG_PATH)
print("\nAll checkpoints are under:", RUNS_DIR)
print("Artifacts are under:", ARTIFACT_DIR)


2026-06-03 04:31:19,219 | INFO | PROJECT_ROOT=/home/diy-hus/fruitnet-chili-yield
2026-06-03 04:31:19,219 | INFO | DATA_ROOT=/mnt/f/fruitnet-chili-yield-data
2026-06-03 04:31:19,220 | INFO | OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs
2026-06-03 04:31:19,220 | INFO | COCO_PROC=/mnt/f/fruitnet-chili-yield-data/processed/coco_yolo
2026-06-03 04:31:19,220 | INFO | RUNS_DIR=/mnt/f/fruitnet-chili-yield-outputs/runs/detectors/coco_pretrain
2026-06-03 04:31:19,220 | INFO | ARTIFACT_DIR=/mnt/f/fruitnet-chili-yield-outputs/artifacts/pretrained
2026-06-03 04:31:19,221 | INFO | RESULT_DIR=/mnt/f/fruitnet-chili-yield-outputs/results/exp00_coco_pretrain
2026-06-03 04:31:19,222 | INFO | Python=3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
2026-06-03 04:31:19,222 | INFO | Platform=Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
2026-06-03 04:31:19,222 | INFO | Log file=/mnt/f/fruitnet-chili-yield-outputs/logs/02_coco_pretrain_resume_safe_20260603_043119.log
2026-06-03 04:31:19,22


Summary: /mnt/f/fruitnet-chili-yield-outputs/results/exp00_coco_pretrain/coco_pretrain_summary.csv
      model_key  status                                        resume_from  \
0       yolov5s      ok  /mnt/f/fruitnet-chili-yield-outputs/runs/detec...   
1       yolov8n      ok                                                      
2    fruitnet_b  failed                                                      
3    fruitnet_g  failed                                                      
4   fruitnet_gc  failed                                                      
5  fruitnet_gcs  failed                                                      

                                             best_pt  \
0  /mnt/f/fruitnet-chili-yield-outputs/runs/detec...   
1  /mnt/f/fruitnet-chili-yield-outputs/runs/detec...   
2                                                      
3                                                      
4                                                      
5                